# schedule_model_difference_notify_v1

目标：验证 `src/schedule` 的 dashboard 定时刷新通知能够带出 `model_difference_explanations` 的 top signal、decision、regime segment、winner/loser 和 spread，避免只生成 dashboard 文件而通知里看不到模型差异原因。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

EXPERIMENT_DIR = REPO_ROOT / "research/schedule_model_difference_notify_v1"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

events: list[str] = []

def write_json(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def matrix_runner(**kwargs):
    card_dir = Path(kwargs["card_dir"])
    card_dir.mkdir(parents=True, exist_ok=True)
    write_json(
        card_dir / "analysis_summary.json",
        {"status": "feature_set_matrix_completed", "variant_count": 2, "models": list(kwargs["model_names"])},
    )
    return {"variant_count": 2, "status": "feature_set_matrix_completed"}

def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    payload = {
        "schema_version": "dashboard.model_matrix.v1",
        "artifact_id": "schedule_model_difference_notify_v1",
        "top_models": [{"model": "ridge_mse", "rank": 1}],
        "manifest_source": {
            "kind": "analysis_manifest",
            "path": str(card_dir / "analysis_manifest.json"),
            "schema_version": "artifact.v1",
            "manifest_hash": "f" * 64,
            "producer": "src.analysis.exports",
        },
        "model_difference_explanations": [
            {
                "schema_version": "analysis.model_difference_explanation.v1",
                "priority_rank": 1,
                "segment_type": "window_regime",
                "segment": "stress_window::stress",
                "window": "stress_window",
                "regime": "stress",
                "baseline_model": "ridge_mse",
                "winner_model": "ridge_mse",
                "loser_model": "gcn_mse",
                "relative_return_pct_spread": 6.5,
                "winner_vs_baseline_delta": 0.0,
                "explanation_signal": "baseline_defends",
                "decision_hint": "block_or_rework_loser",
                "review_note": "中文 review：ridge_mse 在 stress 区间相对 gcn_mse 更稳，通知需要直接暴露该差异。",
            }
        ],
    }
    write_json(output_path, payload)
    return payload

config = ModelDashboardPipelineConfig(
    name="model_dashboard_refresh",
    card_dir=EXPERIMENT_DIR / "matrix_card",
    dashboard_output_path=EXPERIMENT_DIR / "dashboard_export.json",
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=("ridge_mse", "gcn_mse"),
    notify_details=True,
)

result = run_model_dashboard_refresh_once(
    config,
    lock_dir=EXPERIMENT_DIR / "locks",
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)

result.status, events[0]


('success',
 'model_dashboard_refresh:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=/Users/rziiiii/dev/workspace/research/schedule_model_difference_notify_v1/dashboard_export.json | model_difference=baseline_defends/block_or_rework_loser stress_window::stress gcn_mse->ridge_mse spread=6.5')

In [2]:
payload_keys = [
    "model_difference_schema_version",
    "model_difference_count",
    "model_difference_top_signal",
    "model_difference_top_decision_hint",
    "model_difference_top_segment",
    "model_difference_top_winner_model",
    "model_difference_top_loser_model",
    "model_difference_top_spread",
]
summary = {
    "status": result.status,
    "notification": events[0],
    "payload_extract": {key: result.payload.get(key) for key in payload_keys},
    "sidecars": {
        "dashboard_export": str(EXPERIMENT_DIR / "dashboard_export.json"),
        "notification_events": str(EXPERIMENT_DIR / "notification_events.json"),
        "analysis_summary": str(EXPERIMENT_DIR / "analysis_summary.json"),
    },
}
write_json(EXPERIMENT_DIR / "notification_events.json", events)
write_json(EXPERIMENT_DIR / "analysis_summary.json", summary)

conclusion = """# 中文结论

- 做了什么：用最小 dashboard payload 验证 `src/schedule` 在 `notify_details=True` 时读取 `model_difference_explanations`。
- 核心指标：`model_difference_count=1`，top signal 为 `baseline_defends`，decision 为 `block_or_rework_loser`，segment 为 `stress_window::stress`，spread 为 `6.5`。
- regime 差异：在 `stress_window::stress` 区间，`ridge_mse` 相对 `gcn_mse` 是 winner，说明定时通知可以直接暴露区间模型差异。
- 是否晋升：仅为 schedule 通知能力证据，可随本次 `src/schedule` 改动进入 production 框架；不代表策略晋升。
- 风险：本实验使用 mock dashboard export，只验证通知字段传递，不验证真实回测指标质量。
- 下一步：接入真实 dashboard export 后，可在通知系统中按 signal/decision 分流提醒。"""

(EXPERIMENT_DIR / "analysis_conclusion.md").write_text(conclusion, encoding="utf-8")
summary


{'status': 'success',
 'notification': 'model_dashboard_refresh:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=/Users/rziiiii/dev/workspace/research/schedule_model_difference_notify_v1/dashboard_export.json | model_difference=baseline_defends/block_or_rework_loser stress_window::stress gcn_mse->ridge_mse spread=6.5',
 'payload_extract': {'model_difference_schema_version': 'analysis.model_difference_explanation.v1',
  'model_difference_count': 1,
  'model_difference_top_signal': 'baseline_defends',
  'model_difference_top_decision_hint': 'block_or_rework_loser',
  'model_difference_top_segment': 'stress_window::stress',
  'model_difference_top_winner_model': 'ridge_mse',
  'model_difference_top_loser_model': 'gcn_mse',
  'model_difference_top_spread': 6.5},
 'sidecars': {'dashboard_export': '/Users/rziiiii/dev/workspace/research/schedule_model_difference_notify_v1/dashboard_export.json',
  'notification_events': '/Users/rziiiii/dev/workspace/r

# 中文结论

- 做了什么：验证 `src/schedule` 的 dashboard refresh 在 `notify_details=True` 时，把 `model_difference_explanations` 接进结果 payload 和通知文本。
- 核心指标：期望通知出现 `baseline_defends/block_or_rework_loser`、`stress_window::stress`、`gcn_mse->ridge_mse` 和 `spread=6.5`。
- regime 差异：stress regime 下 `ridge_mse` 是 winner，`gcn_mse` 是 loser，通知需要直接呈现该差异，方便 review。
- 是否晋升：这是框架能力验证，可随 `src/schedule` 改动进入 production；不是单个交易策略晋升。
- 风险：本 notebook 使用 mock 数据验证 schedule contract，不替代真实模型回测。
- 下一步：真实 dashboard export 生成后，通知端可以按 decision hint 做分流。